# Exploración SESCO — Producción de petróleo y gas (Argentina)

Notebook exploratoria para el MVP **Oil & Gas Dashboard**.

- Fuente CKAN: [energia-produccion-petroleo-gas-sesco](https://datos.gob.ar/dataset/energia-produccion-petroleo-gas-sesco)
- Objetivo: entender recursos, columnas y calidad de datos antes de API/DB/dashboard.
- `backend/` y `frontend/` quedan pausados durante esta etapa.

## 1. Importar librerías

In [ ]:
import re
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 50)

# Rutas relativas al directorio exploration/
NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name == "notebooks":
    EXPLORATION_DIR = NOTEBOOK_DIR.parent
else:
    EXPLORATION_DIR = NOTEBOOK_DIR / "exploration"

RAW_DIR = EXPLORATION_DIR / "data" / "raw"
PROCESSED_DIR = EXPLORATION_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Exploration dir: {EXPLORATION_DIR}")

## 2. Consultar API CKAN

In [ ]:
PACKAGE_ID = "energia-produccion-petroleo-gas-sesco"
CKAN_URL = f"https://datos.gob.ar/api/3/action/package_show?id={PACKAGE_ID}"
HEADERS = {"User-Agent": "OilGas-Exploration/1.0"}

response = requests.get(CKAN_URL, headers=HEADERS, timeout=60)
response.raise_for_status()
payload = response.json()

print(f"HTTP status: {response.status_code}")
print(f"success: {payload.get('success')}")

## 3. Validar respuesta

In [ ]:
assert payload.get("success") is True, "CKAN no devolvió success=True"
assert "result" in payload, "Falta clave 'result' en la respuesta"

package = payload["result"]
resources = package.get("resources", [])

print(f"Dataset: {package.get('title')}")
print(f"Recursos: {len(resources)}")
print(f"Última modificación del paquete: {package.get('metadata_modified')}")

## 4–6. DataFrame de recursos y listado ordenado

In [ ]:
resource_fields = ["name", "id", "format", "url", "created", "last_modified", "size"]

df_resources = pd.DataFrame(
    [{field: resource.get(field) for field in resource_fields} for resource in resources]
)

df_resources = df_resources.sort_values("name", key=lambda s: s.str.lower()).reset_index(drop=True)
df_resources

## 7. Recursos candidatos por palabras clave

In [ ]:
KEYWORDS = [
    "petroleo", "petróleo", "gas", "provincia", "cuenca",
    "empresa", "sesco", "shale", "tight",
]


def normalize_text(value: str) -> str:
    """Minúsculas sin acentos para búsqueda flexible."""
    if not isinstance(value, str):
        return ""
    text = unicodedata.normalize("NFKD", value)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    return text.lower()


def match_keywords(name: str, keywords: list[str]) -> list[str]:
    normalized = normalize_text(name)
    return [kw for kw in keywords if normalize_text(kw) in normalized]


df_candidates = df_resources.copy()
df_candidates["keywords_matched"] = df_candidates["name"].apply(
    lambda name: match_keywords(name, KEYWORDS)
)
df_candidates = df_candidates[df_candidates["keywords_matched"].map(len) > 0]
df_candidates[["name", "format", "keywords_matched", "url"]]

## 8. Identificar recursos iniciales para el MVP

In [ ]:
MVP_TARGETS = {
    "petroleo_provincia": ["petroleo", "provincia"],
    "gas_provincia": ["gas", "provincia"],
    "petroleo_cuenca": ["petroleo", "cuenca"],
    "gas_cuenca": ["gas", "cuenca"],
    "petroleo_empresa": ["petroleo", "empresa"],
    "gas_empresa": ["gas", "empresa"],
}


def find_mvp_resource(target_keywords: list[str]) -> pd.Series | None:
    """Busca el recurso CSV más reciente que contenga todas las palabras clave."""
    matches = []
    for _, row in df_resources.iterrows():
        normalized = normalize_text(row["name"])
        if all(normalize_text(kw) in normalized for kw in target_keywords):
            matches.append(row)
    if not matches:
        return None

    df_match = pd.DataFrame(matches)
    # Preferir CSV y recursos de producción promedio diaria
    df_match["is_csv"] = df_match["format"].str.upper().eq("CSV")
    df_match["is_promedio"] = df_match["name"].apply(
        lambda x: "promedio" in normalize_text(x)
    )
    df_match = df_match.sort_values(
        ["is_csv", "is_promedio", "last_modified"],
        ascending=[False, False, False],
    )
    return df_match.iloc[0]


mvp_resources = {}
for label, keywords in MVP_TARGETS.items():
    match = find_mvp_resource(keywords)
    mvp_resources[label] = match
    status = "OK" if match is not None else "NO ENCONTRADO"
    name = match["name"] if match is not None else "—"
    print(f"[{status}] {label}: {name}")

df_mvp = pd.DataFrame(
    {
        "mvp_target": list(mvp_resources.keys()),
        "found": [v is not None for v in mvp_resources.values()],
        "name": [v["name"] if v is not None else None for v in mvp_resources.values()],
        "url": [v["url"] if v is not None else None for v in mvp_resources.values()],
    }
)
df_mvp

## 9. Descargar un CSV candidato

In [ ]:
# Recurso principal para esta exploración: petróleo promedio diario por provincia
selected_resource = mvp_resources.get("petroleo_provincia")
if selected_resource is None:
    selected_resource = df_resources[
        df_resources["format"].str.upper().eq("CSV")
    ].iloc[0]

download_url = selected_resource["url"]
source_resource_name = selected_resource["name"]
local_filename = "produccion_petroleo_promedio_diaria_por_provincia.csv"
local_path = RAW_DIR / local_filename

print(f"Recurso: {source_resource_name}")
print(f"URL: {download_url}")

if not local_path.exists():
    file_response = requests.get(download_url, headers=HEADERS, timeout=120)
    file_response.raise_for_status()
    local_path.write_bytes(file_response.content)
    print(f"Descargado en: {local_path}")
else:
    print(f"Ya existe: {local_path}")

## 10–11. Leer CSV e inspeccionar estructura

In [ ]:
df_raw = pd.read_csv(local_path, encoding="utf-8-sig")
print(f"Filas: {len(df_raw):,} | Columnas: {len(df_raw.columns)}")
df_raw.head(10)

In [ ]:
print("Columnas:", list(df_raw.columns))
print("\nTipos de datos:")
print(df_raw.dtypes)

print("\nValores nulos:")
print(df_raw.isna().sum())

print("\nCardinalidad por columna:")
for col in df_raw.columns:
    print(f"  {col}: {df_raw[col].nunique()} únicos")

In [ ]:
# Rango temporal si existen columnas de fecha/período/año/mes
date_candidates = [c for c in df_raw.columns if any(
    token in normalize_text(c) for token in ["anio", "ano", "mes", "fecha", "periodo", "indice", "tiempo"]
)]
print("Columnas temporales detectadas:", date_candidates)

if "indice_tiempo" in df_raw.columns:
    print(f"Rango indice_tiempo: {df_raw['indice_tiempo'].min()} → {df_raw['indice_tiempo'].max()}")
if "anio" in df_raw.columns:
    print(f"Rango anio: {df_raw['anio'].min()} → {df_raw['anio'].max()}")
if "mes" in df_raw.columns:
    print(f"Rango mes: {df_raw['mes'].min()} → {df_raw['mes'].max()}")

## 12. Funciones auxiliares de inspección

In [ ]:
def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nombres de columnas a snake_case sin acentos."""
    def clean(name: str) -> str:
        text = unicodedata.normalize("NFKD", str(name))
        text = "".join(ch for ch in text if not unicodedata.combining(ch))
        text = text.strip().lower()
        text = re.sub(r"[^a-z0-9]+", "_", text)
        return text.strip("_")

    out = df.copy()
    out.columns = [clean(col) for col in out.columns]
    return out


def detect_date_columns(df: pd.DataFrame) -> list[str]:
    tokens = ("anio", "ano", "mes", "fecha", "periodo", "indice", "tiempo")
    return [col for col in df.columns if any(token in col for token in tokens)]


def detect_numeric_columns(df: pd.DataFrame) -> list[str]:
    return df.select_dtypes(include="number").columns.tolist()


def detect_categorical_columns(df: pd.DataFrame) -> list[str]:
    numeric = set(detect_numeric_columns(df))
    date_like = set(detect_date_columns(df))
    return [col for col in df.columns if col not in numeric and col not in date_like]


def summarize_dataframe(df: pd.DataFrame) -> dict:
    return {
        "rows": len(df),
        "columns": len(df.columns),
        "date_columns": detect_date_columns(df),
        "numeric_columns": detect_numeric_columns(df),
        "categorical_columns": detect_categorical_columns(df),
        "null_counts": df.isna().sum().to_dict(),
    }


df_norm = normalize_column_names(df_raw)
summary = summarize_dataframe(df_norm)
summary

## 13. Normalización tentativa hacia modelo analítico

In [ ]:
TARGET_SCHEMA = [
    "periodo",
    "anio",
    "mes",
    "producto",
    "agrupador_tipo",
    "agrupador_nombre",
    "tipo_recurso",
    "produccion",
    "source_resource",
]


def infer_product(name: str) -> str:
    n = normalize_text(name)
    if "petroleo" in n or "petr" in n:
        return "petroleo"
    if "gas" in n:
        return "gas"
    return "desconocido"


def infer_group_type(name: str) -> str:
    n = normalize_text(name)
    for label in ("provincia", "cuenca", "empresa", "yacimiento", "pais"):
        if label in n:
            return label
    return "otro"


def infer_tipo_recurso(name: str) -> str | None:
    n = normalize_text(name)
    if "shale" in n or "tight" in n:
        return "shale_tight"
    if "promedio" in n:
        return "promedio_diario"
    if "historica" in n or "serie" in n:
        return "serie_historica"
    return None


def find_production_column(columns: list[str]) -> str | None:
    for col in columns:
        if "produccion" in col or col.startswith("prod"):
            return col
    numeric = detect_numeric_columns(df_norm)
    excluded = {"anio", "mes"}
    candidates = [c for c in numeric if c not in excluded]
    return candidates[0] if candidates else None


def find_group_column(columns: list[str], group_type: str) -> str | None:
    for col in columns:
        if group_type in col:
            return col
    cat_cols = detect_categorical_columns(df_norm)
    return cat_cols[0] if cat_cols else None


producto = infer_product(source_resource_name)
agrupador_tipo = infer_group_type(source_resource_name)
tipo_recurso = infer_tipo_recurso(source_resource_name)
prod_col = find_production_column(list(df_norm.columns))
group_col = find_group_column(list(df_norm.columns), agrupador_tipo)

df_model = pd.DataFrame()
df_model["periodo"] = df_norm["indice_tiempo"] if "indice_tiempo" in df_norm.columns else None
df_model["anio"] = df_norm["anio"] if "anio" in df_norm.columns else None
df_model["mes"] = df_norm["mes"] if "mes" in df_norm.columns else None
df_model["producto"] = producto
df_model["agrupador_tipo"] = agrupador_tipo
df_model["agrupador_nombre"] = df_norm[group_col] if group_col else None
df_model["tipo_recurso"] = tipo_recurso
df_model["produccion"] = df_norm[prod_col] if prod_col else None
df_model["source_resource"] = source_resource_name

print("Mapeo tentativo:")
print(f"  producto={producto}, agrupador_tipo={agrupador_tipo}, tipo_recurso={tipo_recurso}")
print(f"  produccion ← {prod_col}, agrupador_nombre ← {group_col}")
df_model.head(10)

## 14. Gráficos exploratorios

In [ ]:
if df_model["periodo"].notna().any() and df_model["produccion"].notna().any():
    ts = (
        df_model.groupby("periodo", as_index=False)["produccion"]
        .sum()
        .sort_values("periodo")
    )

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(ts["periodo"], ts["produccion"], marker="o", linewidth=1.5)
    ax.set_title(f"Evolución temporal — {producto} ({agrupador_tipo})")
    ax.set_xlabel("Período")
    ax.set_ylabel("Producción (suma por período)")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No hay columnas suficientes para serie temporal.")

In [ ]:
if df_model["agrupador_nombre"].notna().any() and df_model["produccion"].notna().any():
    latest_period = df_model["periodo"].max()
    ranking = (
        df_model[df_model["periodo"] == latest_period]
        .groupby("agrupador_nombre", as_index=False)["produccion"]
        .sum()
        .sort_values("produccion", ascending=False)
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(ranking["agrupador_nombre"], ranking["produccion"])
    ax.invert_yaxis()
    ax.set_title(f"Ranking por {agrupador_tipo} — período {latest_period}")
    ax.set_xlabel("Producción")
    plt.tight_layout()
    plt.show()
else:
    print("No hay columnas suficientes para ranking.")

In [ ]:
# Comparación entre agrupadores en los últimos 12 períodos
if df_model["periodo"].notna().any():
    recent_periods = sorted(df_model["periodo"].dropna().unique())[-12:]
    top_groups = (
        df_model[df_model["periodo"] == df_model["periodo"].max()]
        .groupby("agrupador_nombre")["produccion"]
        .sum()
        .sort_values(ascending=False)
        .head(5)
        .index
    )

    compare = df_model[
        df_model["periodo"].isin(recent_periods)
        & df_model["agrupador_nombre"].isin(top_groups)
    ]

    pivot = compare.pivot_table(
        index="periodo",
        columns="agrupador_nombre",
        values="produccion",
        aggfunc="sum",
    ).sort_index()

    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(12, 5))
        pivot.plot(ax=ax, marker="o")
        ax.set_title(f"Top 5 {agrupador_tipo}s — últimos 12 períodos")
        ax.set_xlabel("Período")
        ax.set_ylabel("Producción")
        ax.tick_params(axis="x", rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("Pivot vacío para comparación.")
else:
    print("Sin columna periodo para comparación.")

## 15. Conclusiones (completar tras ejecutar la notebook)

In [ ]:
print("=" * 70)
print("CONCLUSIONES EXPLORATORIAS")
print("=" * 70)

print("\n1) Recursos útiles para el MVP")
for _, row in df_mvp.iterrows():
    flag = "✓" if row["found"] else "✗"
    print(f"   {flag} {row['mvp_target']}: {row['name']}")

print("\n2) Columnas reales del recurso descargado")
print(f"   {list(df_raw.columns)}")

print("\n3) Problemas de limpieza detectados")
issues = []
if df_raw.isna().any().any():
    issues.append(f"Valores nulos en: {df_raw.columns[df_raw.isna().any()].tolist()}")
if any(str(c).startswith("\ufeff") for c in df_raw.columns):
    issues.append("Posible BOM en encabezados (usar encoding='utf-8-sig')")
if not issues:
    issues.append("Sin problemas graves evidentes en esta muestra")
for item in issues:
    print(f"   - {item}")

print("\n4) Datos recomendados para primer dashboard Streamlit")
print("   - Producción promedio diaria por provincia/cuenca/empresa (CSV mensuales)")
print("   - Series SESCO + Tight/Shale por dimensión para comparar convencional vs no convencional")
print(f"   - Recurso analizado hoy: {source_resource_name}")

print("\n5) Dudas abiertas")
print("   - Unidades exactas y definiciones oficiales de cada columna de producción")
print("   - Solapamiento entre recursos 'promedio diaria' y 'SESCO + Tight y Shale'")
print("   - Frecuencia de actualización y lag de publicación")
print("   - Tratamiento de 'Estado Nacional' y categorías administrativas especiales")
print("   - Estrategia de unión temporal entre series anteriores a 2009 y posteriores")